# Lesson 4.4: Selecting, Filtering, and Aggregating

**Analyst:** Meron Welderufael
**Date:** 11/2/2026
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** Practice column selection, boolean filtering, and
groupby aggregation on the 200,000-row FinFlow dataset. Compute the
overall churn rate. Preview the per-tier churn breakdown.
**Task** You're producing a clean per-tier churn report that goes into the Friday memo. Five questions, each with a one-line or two-line Pandas answer.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('finflow-customers-200k.csv',
                 parse_dates = ['signup_date', 'last_login_date', 'churn_date']
                )
print(f'Shape: {df.shape}')

Shape: (200000, 17)


## Question 1: Per-tier customer counts.
How many customers does FinFlow have in each plan tier? Use value_counts() or a groupby with .size(). Print the result.

In [2]:
customers_by_tier = df.groupby('plan_tier').size()
print(f'Customers in each plan tier')
print(customers_by_tier)

Customers in each plan tier
plan_tier
Basic         120016
Enterprise     20106
Pro            59878
dtype: int64


## Question 2: Per-tier churn rate.
What is the churn rate (proportion of customers where active == False) in each plan tier? Express as a percentage. Use the groupby + boolean-mean pattern from Step 11 of the Guided Example.

In [3]:
active_rate_by_tier = df.groupby('plan_tier')['active'].mean()
print('Active rate by plan tier:')
print(active_rate_by_tier)
print()
churn_rate_by_tier = (1 - df.groupby('plan_tier')['active'].mean())*100
print("Churn rate by tier:")
print(churn_rate_by_tier)

Active rate by plan tier:
plan_tier
Basic        0.75
Enterprise   0.95
Pro          0.85
Name: active, dtype: float64

Churn rate by tier:
plan_tier
Basic        25.08
Enterprise    4.96
Pro          14.86
Name: active, dtype: float64


## Question 3: Per-tier monthly recurring revenue from active customers.
What is the sum of monthly_fee for active customers in each plan tier? Hint: filter to active customers first, then groupby.

In [4]:
active_tier = df[df['active']]
active_by_tier = active_tier.groupby('plan_tier')['monthly_fee'].sum()
print(active_by_tier)

plan_tier
Basic         898220.88
Enterprise   3802492.00
Pro          1528950.18
Name: monthly_fee, dtype: float64


## Question 4: Per-tier average sessions and support tickets, split by active state.
For each combination of (plan_tier, active), compute the mean of total_sessions and the mean of support_tickets_opened. Use a multi-column groupby with .agg(). Unstack the result so the table reads cleanly.

In [5]:
two_ways = df.groupby(['plan_tier', 'active']).agg({
    'total_sessions': 'mean',
    'support_tickets_opened': 'mean'
}).unstack()
print("For two ways of groupby")
print(two_ways)

For two ways of groupby
           total_sessions        support_tickets_opened      
active              False  True                   False True 
plan_tier                                                    
Basic               25.41 210.32                   4.02  2.00
Enterprise          25.53 210.40                   3.95  2.02
Pro                 25.63 210.00                   4.02  2.00


## Question 5: A consolidated summary table.

Build a single DataFrame that has one row per plan_tier and the following columns:

customer_count
churn_rate_pct
active_mrr (total monthly_fee from active customers)
avg_sessions_active (mean total_sessions among active customers)
avg_sessions_churned (mean total_sessions among churned customers)

In [7]:
#1 customer count plan
customer_count = df.groupby('plan_tier')['customer_id'].count()

#2 churn rate by plan
def churn_rate_pct(active_series):
    return (~active_series).mean() * 100

churn_rate = (
    df.groupby('plan_tier')['active']
    .apply(churn_rate_pct)
)
#3 Active MRR by plan
active_mrr = (
    df[df['active']]
    .groupby('plan_tier')['monthly_fee']
    .sum()
)
#4 Average sessions among active customers 
avg_sessions_active = (
    df[df['active']]
    .groupby('plan_tier')['total_sessions']
    .mean()
)
#5 Average sessions among churned customers
avg_sessions_churned = (
    df[~df['active']]
    .groupby('plan_tier')['total_sessions']
    .mean()
)

# 6. Combine everything into one DataFrame
final_summary = pd.DataFrame({
     'customer_count': customer_count,
     'churn_rate_pct': churn_rate,
     'active_mrr': active_mrr,
     'avg_sessions_active': avg_sessions_active,
     'avg_sessions_churned': avg_sessions_churned
})
   
print(final_summary)

            customer_count  churn_rate_pct  active_mrr  avg_sessions_active  \
plan_tier                                                                     
Basic               120016           25.08   898220.88               210.32   
Enterprise           20106            4.96  3802492.00               210.40   
Pro                  59878           14.86  1528950.18               210.00   

            avg_sessions_churned  
plan_tier                         
Basic                      25.41  
Enterprise                 25.53  
Pro                        25.63  


**Which plan tier has the highest churn rate? Does that match your expectation?** 
The Basic plan has the highest churn rate at 25.08%, followed by Pro at 14.86% and Enterprise at 4.96%. This matches my expectation because we designed the dataset with an expected 25% churn rate for Basic customers. 
**Which plan tier contributes the most active monthly revenue? Is the highest-revenue tier the same as the largest-customer-count tier?**
Enterprise contributes the most active monthly revenue, approximately $3.8 million, even though it has the smallest customer base of 20,106 customers, while Basic has the largest customer base but generates less revenue. This suggests that Enterprise customers generate significantly more revenue per customer. 
**How does average session count compare between active and churned customers in the same tier? What does that pattern suggest for the broader analysis?**
Regarding customer engagement, active customers average approximately 210 sessions, compared to only 25 sessions for churned customers across all three tiers. This significant difference suggests that lower customer engagement may be associated with a higher likelihood of churn, which FinFlow should investigate further to improve customer retention.